# Literature Review Agent
### Description
- Building a multiple agentic system for help researchers to save some time in literature. 

## Agents and their work
|Agents & Tool|Work description|
|------------|---------------|
|Search_Agent| The agent search research paper to the user's topic|
|Search_Tool| The tools help the agent to search the paper using the keywords|
|Downloader| The tools used to download those searched papers|
|DB_Agent| The agent read and divide into data and save those in a vector database|
|Question_Agent| THe agent generate research questions|
|Answer_agent| The agent answer those research question|
|Synthesis_Agent| Finalize the review|

### Dependencies
```bash
pip install langgraph langchain-groq langchain-openrouter chromadb \
            pypdf arxiv duckduckgo-search streamlit python-dotenv
```



## Agent : 1 - Search agent
- This agent will get topic from users input and find keywords and start searching related 

In [42]:
# Calling LLM and setup API
import os
import langchain_openrouter
import langchain_groq
from dotenv import load_dotenv

# Run API key from .env file
load_dotenv()
GROQ_API = os.getenv("GROQ_API")
OPEN_ROUTER_API = os.getenv("OPEN_ROUTER_API")

if not GROQ_API:
    raise ValueError("API keys for GROQ must be set in the .env file.")
else:
    print("GROQ API key is set.")
if not OPEN_ROUTER_API:
    raise ValueError("API keys for OPEN_ROUTER must be set in the .env file.")
else:
    print("OPEN_ROUTER API key is set.")


GROQ API key is set.
OPEN_ROUTER API key is set.


In [45]:
from langchain_groq import ChatGroq
from langchain_openrouter import ChatOpenRouter
# Search agent
llm_search = ChatGroq(model="openai/gpt-oss-20b", 
                      api_key=GROQ_API, 
                      temperature=0, 
                      max_tokens=1000)
# Testing
response = llm_search.invoke("What is Tuberculosis?")
print(f"LLM Read for search agent with model: {llm_search.model}")
print(f"Response: {response}")



LLM Read for search agent with model: openai/gpt-oss-20b
Response: content='**Tuberculosis (TB)** is a contagious infectious disease caused by the bacterium *Mycobacterium tuberculosis*. It most commonly affects the lungs (pulmonary TB) but can involve any organ (extrapulmonary TB).\n\n---\n\n## 1. How TB Spreads\n- **Airborne transmission**: When an infected person coughs, sneezes, sings, or speaks, tiny droplets containing the bacteria are released into the air.  \n- **Inhalation**: A susceptible person inhales these droplets and the bacteria can settle in the lungs.  \n- **Latency**: Most people who inhale the bacteria do not develop active disease immediately; the infection can remain dormant (latent TB) for years.\n\n---\n\n## 2. Clinical Forms\n\n| Form | Typical Features | Common Sites |\n|------|------------------|--------------|\n| **Latent TB infection (LTBI)** | No symptoms; bacteria are present but inactive | – |\n| **Pulmonary TB** | Cough (often >3\u202fweeks), chest pain

In [46]:
# Search query & tool implementation
import requests
import feedparser
def gen_search_queries(topic, n=3):
    prompt = f"""You are a research assistant. Generate {n} search queries for the topic: "{topic}".
    Each query should be concise and relevant to the topic. Return the queries as a list of strings.
    Return only numbered list of queries without any additional text or explanation.
    
    Topic:{topic}
    """
    response = llm_search.invoke(prompt) 
    lines = [line.strip() for line in response.content.split("\n") if line.strip()]
    queries =[]
    for line in lines:
        if line[0].isdigit():
            q = line.split('.',1)[1].strip()
            queries.append(q)
        return queries[:n] if queries else [topic]
    
def search_arxiv(query, max_results=5):
    base_url = "http://export.arxiv.org/api/query"
    params = {
        "search_query": f"all:{query}",
        "start": 0,
        "max_results": max_results,
        "sortBy": "relevance",
        "sortOrder": "descending"
    }
    try:
        response = requests.get(base_url, params=params, timeout=20)
        feed = feedparser.parse(response.text)
    except Exception as e:
        print(f"arxiv search failed for query '{query}': {e}")
        return []
    
    results = []
    for entry in feed.entries:
        result = {
            "title": entry.title.replace('\n', ' ').strip(), # Remove newlines and extra spaces
            "authors":[a.name for a in entry.authors],
            "abstract": entry.summary.replace('\n', ' ').strip(), 
            "pdf_url": entry.published,
            "source": "arxiv"
        }
        results.append(result)
    return results
def search_semantic_scholar(query, max_results=5):
    url = "https://api.semanticscholar.org/graph/v1/paper/search"
    params = {
        "query": query,
        "limit": max_results,
        "fields": "title,abstract,authors,url,openAccessPdf,year"
    }
    try:
        response = requests.get(url, params=params, timeout=15)
        data = response.json()
    except Exception as e:
        print(f"Semantic Scholar search failed for '{query}': {e}")
        return []

    results = []
    for paper in data.get("data", []):
        results.append({
            "title": paper.get("title"),
            "authors": [a["name"] for a in paper.get("authors", [])],
            "abstract": paper.get("abstract") or "",
            "pdf_url": paper.get("openAccessPdf", {}).get("url") if paper.get("openAccessPdf") else None,
            "published": paper.get("year"),
            "source": "semantic_scholar"
        })
    return results


    

